# Species distribution modeling with multimodal satellite and environmental data

Welcome to this wonderfull main to discover the beauty of mother earth, enjoy your work and be happy :)

## 1. Setup

In [1]:
from pathlib import Path
try:
    root = Path(__file__).resolve().parent
except NameError:
    root = Path.cwd()  # fallback for Jupyter notebooks

while root.parent != root:
    if any((root / marker).exists() for marker in ["README.md"]):
        break
    root = root.parent

# Fallback in case nothing found
if not any((root / marker).exists() for marker in ["README.md"]):
    print("Could not locate project root — defaulting to current working directory")
    root = Path.cwd()

root = str(root)
print(f"Root folder detected at: {root}")

Root folder detected at: /Users/todorovkatia/Documents/EPFL/MA3/IPEO/Projet/IPEO-Species-distribution


In [2]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
import torch.nn as nn
import satlaspretrain_models 
from satlaspretrain_models import Model
from torch.utils.data import dataset
from torch.utils.data import DataLoader
import numpy as np
import os

## 2. Data loading

In [4]:
from Aurelien_DataLoader.GeoPlantDataset import GeoPlantDataset, viz_sample
train=GeoPlantDataset(data_folder=f"{root}/data", split='train')

def load_dataloader(batch_size, split='train'):
  return DataLoader(
      GeoPlantDataset(data_folder=f"{root}/data", split=split),
      batch_size=batch_size,
      shuffle=(split=='train'),       # we shuffle the image order for the training dataset
      num_workers=2                   # perform data loading with two CPU threads
  )

## 3. Create model

In [23]:
# neural network with 3 NN (for each modality) and a classifier at the end #same architecture as ex7 and ex9
#maybe add one layer in the CNN for time series?

class multimodal_SDM (nn.Module): #heritates from class nn.Module
    
    def __init__(self, dim_NN_env=128, dim_NN_sat=256, dim_NN_timeseries=64): #dimension of output of neural networks
        super(multimodal_SDM, self).__init__()  #call the init of the parent class
        
        self.dim_NN_env=dim_NN_env

        self.dim_NN_sat=dim_NN_sat

        self.dim_NN_timeseries=dim_NN_timeseries
        
        self.MLP_env = nn.Sequential( #19 bioclim variables to 128 values
            nn.Linear(19,50),
            nn.ReLU(),
            nn.Linear(50, 50),
            nn.ReLU(),
            nn.Linear(50, 50),
            nn.ReLU(),
            nn.Linear(50, dim_NN_env),
            nn.ReLU()
            )
        
        weights_manager = satlaspretrain_models.Weights()
        self.CNN_sat = weights_manager.get_pretrained_model("Sentinel2_Resnet50_SI_RGB", fpn=True, head=satlaspretrain_models.Head.CLASSIFY, 
                                                num_categories=self.dim_NN_sat, device='cpu')
        # self.CNN_sat = Model(weights=torch.load("sentinel2_resnet50_si_rgb.pth", map_location=torch.device('cpu')), 
        #                      backbone=satlaspretrain_models.Backbone.RESNET50, 
        #                      fpn=True, head=satlaspretrain_models.Head.CLASSIFY, num_categories=self.dim_NN_env)


        self.CNN_timeseries= nn.Sequential(
            #R G B NIR with 10 years and 4 seasons= 40 values: 4 channels, length 40
            #Like Alexnet but shorter (2 convolutionnal layers) and in 1D
            nn.Conv1d(in_channels=4, out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(32*10, dim_NN_timeseries), # 32 channels * length 10 (2 poolings of size 2 equivalent to divide /4 the length of the input) ?
            nn.ReLU()
        )

        self.classifier= nn.Sequential(
            nn.Linear(dim_NN_env+dim_NN_sat+dim_NN_timeseries, 1000),
            nn.ReLU(),
            nn.Linear(1000, 342),
            nn.Sigmoid()
        )

    def forward (self,x):
        '''x is a GeoPlantDataset object containing the 3 modalities:
        - env_variables
        - satellite_patches
        - landsat_timeseries
        '''
        env_variables= x['env_variables'].float()
        satellite_patches=x['satellite_patch'].float()
        landsat_timeseries=x['landsat_timeseries'].float()

        #pass each modality through its NN
        NN_env_out=self.MLP_env(env_variables)
        NN_sat_out=self.CNN_sat(satellite_patches)[0]
        NN_time_series_out=self.CNN_timeseries(landsat_timeseries)

        #concatenate the outputs
        combined=torch.cat((NN_env_out, NN_sat_out, NN_time_series_out), dim=1)

        #pass through the classifier
        output=self.classifier(combined)
        return output

In [24]:
#let's test it
dataloader_train = load_dataloader(batch_size=8, split='train')

model = multimodal_SDM() #load the model

#data, _ = iter(dataloader_train).__next__()
data = iter(dataloader_train).__next__()
pred = model(data) #perform 1 forward pass

# assert pred.size(1) == len(train.LABEL_CLASSES), f'ERROR: invalid number of model output channels (should be # classes {len(dataset_train.LABEL_CLASSES)}, got {pred.size(1)})'
# assert pred.size(2) == data.size(2), f'ERROR: invalid spatial height of model output (should be {data.size(2)}, got {pred.size(2)})'
# assert pred.size(3) == data.size(3), f'ERROR: invalid spatial width of model output (should be {data.size(3)}, got {pred.size(3)})'

In [28]:

pred.shape

torch.Size([8, 342])

## 4. Model training

In [ ]:
criterion = nn.functional.binary_cross_entropy

from torch.optim import SGD # stochastic gradient descent

def setup_optimiser(model, learning_rate, weight_decay): # ca sert a quoi de faire une fonction pour mettre un SGD en sortie?
  return SGD(
    model.parameters(), # c'est quoi ca ? 
    learning_rate,
    weight_decay
  )

In [31]:
batch_size = 8
dl_train = load_dataloader(batch_size, 'train')
dl_train

In [ ]:
from tqdm.notebook import trange      # pretty progress bar


def train_epoch(data_loader, model, optimiser, device):

  # set model to training mode. This is important because some layers behave differently during training and testing
  model.train(True)
  model.to(device)

  # stats
  loss_total = 0.0
  oa_total = 0.0

  # iterate over dataset
  pBar = trange(len(data_loader))
  for idx, (data, target) in enumerate(data_loader): 

    # put data and target onto correct device
    data, target = data.to(device), target.to(device)

    # reset gradients
    optimiser.zero_grad()

    # forward pass
    pred = model(data)

    # loss
    loss = criterion(pred, target)

    # backward pass
    loss.backward()

    # parameter update
    optimiser.step()

    # stats update
    loss_total += loss.item()
    oa_total += torch.mean((pred.argmax(1) == target).float()).item()

    # format progress bar
    pBar.set_description('Loss: {:.2f}, OA: {:.2f}'.format(
      loss_total/(idx+1),
      100 * oa_total/(idx+1)
    ))
    pBar.update(1)
  
  pBar.close()

  # normalise stats
  loss_total /= len(data_loader)
  oa_total /= len(data_loader)

  return model, loss_total, oa_total

In [ ]:
def validate_epoch(data_loader, model, device):       # note: no optimiser needed

  # set model to evaluation mode
  model.train(False)
  model.to(device)

  # stats
  loss_total = 0.0
  oa_total = 0.0

  # iterate over dataset
  pBar = trange(len(data_loader))
  for idx, (data, target) in enumerate(data_loader):
    with torch.no_grad():

      # put data and target onto correct device
      data, target = data.to(device), target.to(device)

      # forward pass
      pred = model(data)

      # loss
      loss = criterion(pred, target)

      # stats update
      loss_total += loss.item()
      oa_total += torch.mean((pred.argmax(1) == target).float()).item()

      # format progress bar
      pBar.set_description('Loss: {:.2f}, OA: {:.2f}'.format(
        loss_total/(idx+1),
        100 * oa_total/(idx+1)
      ))
      pBar.update(1)

  pBar.close()

  # normalise stats
  loss_total /= len(data_loader)
  oa_total /= len(data_loader)

  return loss_total, oa_total

In [ ]:
#load and save model

import glob

os.makedirs('cnn_states/multimodal_SDM', exist_ok=True)

def load_model(epoch='latest'):
  model = multimodal_SDM()
  modelStates = glob.glob('cnn_states/multimodal_SDM/*.pth')
  if len(modelStates) and (epoch == 'latest' or epoch > 0):
    modelStates = [int(m.replace('cnn_states/multimodal_SDM/','').replace('.pth', '')) for m in modelStates]
    if epoch == 'latest':
      epoch = max(modelStates)
    stateDict = torch.load(open(f'cnn_states/multimodal_SDM/{epoch}.pth', 'rb'), map_location='cpu')
    model.load_state_dict(stateDict)
  else:
    # fresh model
    epoch = 0
  return model, epoch


def save_model(model, epoch):
  torch.save(model.state_dict(), open(f'cnn_states/multimodal_SDM/{epoch}.pth', 'wb'))

In [ ]:
# define hyperparameters
device = 'cuda'
start_epoch = 0        # set to 0 to start from scratch again or to 'latest' to continue training from saved checkpoint
batch_size = 8
learning_rate = 0.1
weight_decay = 0.001
num_epochs = 30


# initialise data loaders
dl_train = load_dataloader(batch_size, 'train')
dl_val = load_dataloader(batch_size, 'val')

# load model
model, epoch = load_model(epoch=start_epoch)
optim = setup_optimiser(model, learning_rate, weight_decay)

# do epochs
while epoch < num_epochs:

  # training
  model, loss_train, oa_train = train_epoch(dl_train, model, optim, device)

  # validation
  loss_val, oa_val = validate_epoch(dl_val, model, device)

  # print stats
  print('[Ep. {}/{}] Loss train: {:.2f}, val: {:.2f}; OA train: {:.2f}, val: {:.2f}'.format(
      epoch+1, num_epochs,
      loss_train, loss_val,
      100*oa_train, 100*oa_val
  ))

  # save model
  epoch += 1
  save_model(model, epoch)

## 5. Model evaluation

In [ ]:
# we watn computational complexity 